# Лабораторная работа 1 — рабочий ноутбук

**ФИО:**
**Файлы данных:**

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

ROOT_PATH = Path().resolve().parent
DATA_PATH = ROOT_PATH / "data"
CLIENTS_RAW_PATH = DATA_PATH / "clients.csv"
TRANSACTIONS_RAW_PATH = DATA_PATH / "transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [ ]:
df_clients = pd.read_csv(CLIENTS_RAW_PATH, encoding="utf-8")
df_tx = pd.read_csv(TRANSACTIONS_RAW_PATH, encoding="cp1251", sep=";", decimal=",")

df_tx["txn_datetime_parsed"] = pd.to_datetime(df_tx["txn_datetime"], format="mixed", errors="coerce")

In [ ]:
print(f"Shape сlients: {df_clients.shape}")
print(f"Shape transactions: {df_tx.shape}")

display(df_clients.head())
display(df_tx.head())

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [ ]:
print("--- CLIENTS ---")
overview_clients = pd.DataFrame({
    "dtype": df_clients.dtypes.astype(str),
    "n_unique": df_clients.nunique(),
    "n_missing": df_clients.isna().sum(),
    "missing_%": (df_clients.isna().mean() * 100).round(2),
})
display(overview_clients)

print("\n--- TRANSACTIONS ---")
overview_tx = pd.DataFrame({
    "dtype": df_tx.dtypes.astype(str),
    "n_unique": df_tx.nunique(),
    "n_missing": df_tx.isna().sum(),
    "missing_%": (df_tx.isna().mean() * 100).round(2),
})
display(overview_tx)

### 2.2. Дубли

In [ ]:
client_dups = df_clients[df_clients["client_id"].duplicated(False)].sort_values("client_id")
client_conflicts = client_dups.groupby("client_id").nunique()
client_conflict_ids = client_conflicts.index[(client_conflicts > 1).any(axis=1)] if len(client_conflicts) else []

print("Duplicate client_id values:", client_dups["client_id"].nunique())
print("Client IDs with conflicting data:", len(client_conflict_ids))
display(client_conflicts.loc[client_conflict_ids].head())

In [ ]:
tx_dups = df_tx[df_tx["txn_id"].duplicated(False)].sort_values("txn_id")
tx_dup_audit = tx_dups.groupby("txn_id").agg(
    rows=("txn_id","size"),
    client_n=("client_id","nunique"),
    amount_n=("amount","nunique"),
    status_n=("status","nunique"),
    raw_datetime_n=("txn_datetime","nunique"),
    parsed_datetime_n=("txn_datetime_parsed","nunique")
)

print("Duplicate txn_id values:", tx_dups["txn_id"].nunique())
print("Completely duplicated transaction rows:", df_tx.duplicated().sum())
print("txn_id with a discrepancy in the original timestamp record:", (tx_dup_audit.raw_datetime_n > 1).sum())
display(tx_dup_audit.head())

### 2.3. Соответствие данных описанию полей из задания

In [ ]:
print("Transaction date range:", df_tx["txn_datetime_parsed"].min(), "—", df_tx["txn_datetime_parsed"].max())
print("Transactions outside the target period (2024-01-01 .. 2025-06-30):", 
      ((df_tx["txn_datetime_parsed"] < "2024-01-01") | (df_tx["txn_datetime_parsed"] > "2025-06-30 23:59:59")).sum())

In [ ]:
print("amount < 0:", int((df_tx["amount"] < 0).sum()), "| amount == 0:", int((df_tx["amount"] == 0).sum()))

In [ ]:
for col in ["gender", "segment", "loyalty_tier", "card_type", "has_mobile_app", "marketing_opt_in"]:
    display(df_clients[col].value_counts(dropna=False).to_frame("count"))

In [ ]:
for col in ["currency", "category", "channel", "status"]:
    display(df_tx[col].value_counts(False).to_frame("count"))

In [ ]:
print("MCC:", sorted(df_tx["mcc_code"].unique().tolist()))
mcc_by_category = df_tx.groupby("category")["mcc_code"].nunique()
print("Number of distinct MCCs within each original category:")
display(mcc_by_category.rename("n_unique_mcc").to_frame())
print("Categories containing all 9 MCCs:", 
      int((mcc_by_category == 9).sum()), "out of", len(mcc_by_category))

### 2.4. Связь между таблицами

In [ ]:
orphan_tx_count = (~df_tx["client_id"].isin(df_clients["client_id"])).sum()
orphan_clients_unique = df_tx.loc[~df_tx["client_id"].isin(df_clients["client_id"]), "client_id"].nunique()

clients_without_tx = (~df_clients["client_id"].isin(df_tx["client_id"])).sum()

print(f"Transactions with client_id missing from reference: {orphan_tx_count} (unique IDs: {orphan_clients_unique})")
print(f"Clients from reference with no transactions: {clients_without_tx}")

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

In [ ]:
clients = df_clients.copy()
tx = df_tx.drop(columns="txn_datetime_parsed").copy()

for col in ["birth_date","registration_date","card_open_date"]:
    clients[col] = pd.to_datetime(clients[col], format="mixed", dayfirst=True, errors="coerce")
tx["txn_datetime"] = pd.to_datetime(tx["txn_datetime"], format="mixed", errors="coerce")
tx["amount"] = pd.to_numeric(tx["amount"], errors="coerce")
tx["mcc_code"] = pd.to_numeric(tx["mcc_code"], errors="coerce")

print(clients[["birth_date","registration_date","card_open_date"]].dtypes)
print(tx[["txn_datetime","amount"]].dtypes)

### 3.2. Категориальные признаки

In [ ]:
city_map = {
    "москва":"Москва", "г. москва":"Москва",
    "казань":"Казань",
    "екатеринбург":"Екатеринбург", "г.екатеринбург":"Екатеринбург",
    "новосибирск":"Новосибирск", "г. новосибирск":"Новосибирск",
    "н. новгород":"Нижний Новгород", "нижний новгород":"Нижний Новгород",
    "санкт-петербург":"Санкт-Петербург", "г. санкт-петербург":"Санкт-Петербург",
    "санкт петербург":"Санкт-Петербург", "спб":"Санкт-Петербург"
}

clients["city_before"] = clients["city"]
clients["city"] = clients["city"].astype("string").str.strip().str.lower().map(city_map).fillna(clients["city"].astype("string").str.strip())

In [ ]:
category_map = {
    "продукты":"Продукты", "одежда и обувь":"Одежда и обувь",
    "транспорт":"Транспорт", "азс":"АЗС", "аптеки":"Аптеки",
    "кафе и рестораны":"Кафе и рестораны", "кафе/рестораны":"Кафе и рестораны",
    "развлечения и досуг":"Развлечения и досуг", "онлайн-сервисы":"Онлайн-сервисы",
    "связь и интернет":"Связь и интернет"
}

tx["category_before"] = tx["category"]
tx["category"] = tx["category"].astype("string").str.strip().str.lower().map(category_map).fillna(tx["category"].astype("string").str.strip())

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

In [ ]:
missing_profile = pd.DataFrame({
    "n_missing": df_clients.isna().sum(),
    "share_pct": (df_clients.isna().mean()*100).round(2)
})

display(missing_profile.query("n_missing > 0"))
print("No app:", (df_clients.has_mobile_app=="Нет").sum())
print("app_last_login missing:", df_clients.app_last_login.isna().sum())
print("satisfaction_score missing ratio:", f"{df_clients.satisfaction_score.isna().mean():.1%}")
print("monthly_income missing ratio:", f"{df_clients.monthly_income.isna().mean():.1%}")

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

In [ ]:
tx_audit = tx.copy()
amount_stats = tx_audit.groupby(["currency","status"]).amount.agg(
    n="count", median="median", q1=lambda x:x.quantile(.25),
    q3=lambda x:x.quantile(.75), p99=lambda x:x.quantile(.99), max="max", min="min"
)

display(amount_stats.round(2))

In [ ]:
def iqr_flags(s):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    return (s < q1-1.5*(q3-q1)) | (s > q3+1.5*(q3-q1))
tx_audit["amount_iqr_flag"] = tx_audit.groupby("currency").amount.transform(iqr_flags)

print("IQR flags:", int(tx_audit.amount_iqr_flag.sum()), "/", len(tx_audit))
print("Largest positive amounts:"); display(tx_audit.nlargest(10,"amount")[["txn_id","client_id","txn_datetime","amount","currency","status"]])

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

In [ ]:
funnel = []

def add_funnel(name, frame):
    funnel.append({
        "Step": name,
        "Rows": len(frame),
        "Unique txn_id": frame.txn_id.nunique()
    })

In [ ]:
t0 = tx.copy()
add_funnel("After format parsing", t0)

t0["_time_precision"] = t0.txn_datetime.astype(str).str.len()
t1 = (
    t0.sort_values("_time_precision", ascending=False)
      .drop_duplicates("txn_id", keep="first")
      .drop(columns="_time_precision")
)
add_funnel("After txn_id deduplication", t1)

t2 = t1[t1.txn_datetime.between(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-06-30 23:59:59"))]
add_funnel("Within period", t2)

t3 = t2[t2.client_id.isin(set(clients.client_id))]
add_funnel("Exists in reference", t3)

t4 = t3[t3.status == "успешно"]
add_funnel("Successful operations", t4)

display(pd.DataFrame(funnel))

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
log_rows = [
    [
        "client_id",
        "Повторные идентификаторы; 40 полностью одинаковых строк и 15 конфликтующих ID",
        "clients.client_id",
        55,
        "55 повторных строк; 15 ID с конфликтами",
        "дедупликация",
        "одна строка на client_id",
        "Для справочника действующих клиентов нужна одна запись на клиента; при конфликте сохранена первая запись исходного файла, неопределённость раскрыта.",
    ],
    [
        "txn_id",
        "Повторные ID операций; 1 255 точных дублей и 245 различий только в точности времени",
        "transactions.txn_id",
        1500,
        "1500 повторных строк",
        "дедупликация",
        "одна строка на txn_id",
        "Один txn_id трактуется как одна операция; сохраняется наиболее точная временная отметка.",
    ],
    [
        "txn_datetime",
        "86 операций за пределами периода 2024-01-01—2025-06-30",
        "txn_datetime в расчётной выборке",
        86,
        "2023-11-27—2026-08-04",
        "исключено из расчёта",
        "2024-01-01—2025-06-30",
        "Задание явно задаёт границы наблюдения; записи сохранены в исходном источнике, но не участвуют в анализе.",
    ],
    [
        "status",
        "Отклонённые операции и возвраты не отражают успешное использование",
        "transactions.status",
        int((t3.status != "успешно").sum()),
        "отклонено / возврат",
        "исключено из метрики",
        "только успешно",
        "Метрика измеряет состоявшееся использование, а не попытки оплаты и возвратные проводки.",
    ],
    [
        "city",
        "Различия регистра, пробелов, сокращений и префикса «г.»",
        "clients.city",
        int((clients.city_before.astype(str) != clients.city.astype(str)).sum()),
        "17 исходных написаний",
        "нормализация",
        "6 канонических городов",
        "Очевидные варианты сведены к единому написанию без изменения географического смысла.",
    ],
    [
        "category",
        "Варианты написания двух категорий",
        "transactions.category",
        int((tx.category_before.astype(str) != tx.category.astype(str)).sum()),
        "Кафе/рестораны; регистр; Интернет",
        "нормализация",
        "9 категорий",
        "Нужна единая частотная аналитика по категориям.",
    ],
    [
        "amount",
        "Асимметрия и IQR-выбросы по валютам",
        "transactions.amount",
        int(tx_audit.amount_iqr_flag.sum()),
        "Суммы за пределами 1.5 IQR",
        "оставлено",
        "исходные суммы",
        "Без подтверждения ошибки/мошенничества удалять операции нельзя; суммы не входят в основную метрику.",
    ],
    [
        "mcc_code",
        "Описание обещает соответствие MCC категории, но в каждой исходной категории встречаются все 9 MCC",
        "transactions: все строки",
        len(df_tx),
        "9 категорий × 9 MCC",
        "оставлено",
        "MCC не используется для категоризации",
        "Нельзя надёжно восстановить связь категории с MCC; для анализа используется поле category.",
    ],
    [
        "branch_code",
        "Код отделения содержит ведущие нули и не должен интерпретироваться как число",
        "clients.branch_code",
        len(df_clients),
        "Пятизначные коды, например 00145",
        "сохранено как строка",
        "пятизначный строковый код",
        "Числовое чтение теряет ведущие нули; поле сохраняется как идентификатор, а не количественная величина.",
    ],
]

cleaning_log = pd.DataFrame(
    log_rows,
    columns=[
        "field",
        "issue",
        "scope",
        "n_rows",
        "value_before",
        "action",
        "value_after",
        
        "rationale",
    ],
)

display(cleaning_log)

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

In [ ]:
clients["age"] = ((pd.Timestamp("2025-06-30") - clients.birth_date).dt.days / 365.25)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(clients["age"].dropna(), bins=25)
axes[0].set(title="Client Age", xlabel="Age, years", ylabel="Number of Clients")
axes[1].hist(clients["monthly_income"].dropna(), bins=35)
axes[1].set(title="Monthly Income", xlabel="Income, RUB", ylabel="Number of Clients")
plt.tight_layout()
plt.show()
display(clients[["age", "monthly_income", "satisfaction_score"]].describe(percentiles=[.25, .5, .75, .9]).T)

### 5.2. Частотные характеристики категориальных признаков

In [ ]:
cat_cols=["city","segment","loyalty_tier","card_type","has_mobile_app","marketing_opt_in","income_source"]
for c in cat_cols:
    display((clients[c].value_counts(dropna=False).rename("n").to_frame()
             .assign(share_pct=lambda x:(x.n/len(clients)*100).round(1))))

### 5.3. Динамика по месяцам

In [ ]:
activity_tx = t4.copy()
activity_tx["month"] = activity_tx.txn_datetime.dt.to_period("M")
monthly_total = activity_tx.groupby("month").size().reindex(pd.period_range("2024-01", "2025-06", freq="M"), fill_value=0)
monthly_avg = monthly_total / len(clients)

ax = monthly_total.plot(figsize=(12, 4), marker="o")
ax.set(title="Успешные операции по месяцам", xlabel="Месяц", ylabel="Число операций")
ax.legend()
plt.tight_layout()
plt.show()

display(monthly_total.rename("successful_operations").to_frame())

### 5.4. Разрезы и связи между признаками

In [ ]:
clients_unique = clients.drop_duplicates("client_id", keep="first").copy()
periods = pd.period_range("2024-01", "2025-06", freq="M")
monthly_client = activity_tx.assign(month=activity_tx.txn_datetime.dt.to_period("M")).groupby(["client_id", "month"]).size().unstack(fill_value=0)
monthly_client = monthly_client.reindex(index=clients_unique.client_id, columns=periods, fill_value=0)
base_cols = pd.period_range("2024-01", "2024-12", freq="M")
eval_cols = pd.period_range("2025-01", "2025-06", freq="M")

In [ ]:
metrics = pd.DataFrame(index=clients_unique.client_id)
metrics["base_avg"] = monthly_client[base_cols].mean(axis=1)
metrics["eval_avg"] = monthly_client[eval_cols].mean(axis=1)
metrics["base_total"] = monthly_client[base_cols].sum(axis=1)
metrics["eval_total"] = monthly_client[eval_cols].sum(axis=1)
metrics["base_active_months"] = (monthly_client[base_cols] > 0).sum(axis=1)
metrics["eval_active_months"] = (monthly_client[eval_cols] > 0).sum(axis=1)
metrics["drop_pct"] = np.where(metrics.base_avg > 0, 1 - metrics.eval_avg / metrics.base_avg, np.nan)
metrics["drop_abs"] = metrics.base_avg - metrics.eval_avg

profile_cols = ["segment", "loyalty_tier", "card_type", "has_mobile_app", "marketing_opt_in", "city", "branch_code", "income_source", "monthly_income", "satisfaction_score", "app_last_login", "age"]
profile = clients_unique.set_index("client_id")[profile_cols]
metrics = profile.join(metrics)

In [ ]:
for c in ["segment", "loyalty_tier", "card_type", "has_mobile_app", "city"]:
    display(metrics.groupby(c).agg(
        сount=("base_avg", "size"),
        base_monthly=("base_avg", "mean"),
        eval_monthly=("eval_avg", "mean"),
        median_drop=("drop_pct", "median"),
        share_drop_30=("drop_pct", lambda x: (x > .30).mean())
    ).round(3))

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")